# SNAPROAD — Klasifikasi Tingkat Kerusakan Jalan (MobileNetV2 → TensorFlow Lite)

**Mata kuliah:** Pemrograman Bergerak · **Proyek:** SNAPROAD
**Pipeline:** Dataset Kaggle → pra-olah (crop bbox) → transfer learning MobileNetV2 → evaluasi → ekspor `.tflite` → integrasi Flutter

---

## Yang dihasilkan notebook ini

| Berkas | Isi |
|---|---|
| `snaproad_mobilenetv2_fp16.tflite` | Model utama (~2–3 MB) untuk aplikasi Flutter |
| `snaproad_mobilenetv2_int8.tflite` | Varian kuantisasi penuh (opsional, untuk perbandingan di laporan) |
| `labels.txt` | Urutan kelas: `ringan`, `sedang`, `berat` (WAJIB sama dengan aplikasi) |
| `model_meta.json` | Ukuran input, preprocessing, urutan kelas, metrik |
| `confusion_matrix.png`, `training_curves.png`, `class_distribution.png` | Gambar untuk laporan |
| `metrics.json` | accuracy, macro F1, classification report |

## Pemetaan kelas (dataset → aplikasi)

Dataset **RDD2022ES** punya 8 kelas (4 jenis kerusakan × 2 tingkat keparahan). Delapan kelas itu dipetakan ke 3 tingkat kerusakan sesuai Bab 3 dokumen spesifikasi:

| Kelas RDD2022ES | ID | → | Kelas aplikasi |
|---|---|---|---|
| longitudinal crack | 0 | → | ringan |
| transverse crack | 2 | → | ringan |
| longitudinal crack wide | 1 | → | sedang |
| transverse crack wide | 3 | → | sedang |
| alligator crack | 4 | → | sedang |
| pothole | 6 | → | sedang |
| alligator crack sunken | 5 | → | berat |
| pothole deep | 7 | → | berat |

> Pemetaan ini adalah keputusan desain, bukan label bawaan dataset. Tulis alasannya di laporan (Bab 3) dan sebutkan sebagai batasan penelitian.

## Cara pakai di Google Colab

1. Buka [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** → pilih file `.ipynb` ini.
2. **Runtime → Change runtime type → T4 GPU** → Save.
3. Siapkan token Kaggle: buka **kaggle.com → klik foto profil → Settings → API → Create New Token**. Akan terunduh `kaggle.json`.
4. **Runtime → Run all**. Saat diminta, unggah `kaggle.json`.
5. Perkiraan waktu: **35–60 menit** (unduh ±2,6 GB + training 2 tahap di T4).
6. Di akhir, notebook mengunduh `snaproad_model.zip`. Simpan ke folder `ml/artifacts/` di repo.

> Dataset: **RDD2022ES** (`juusos/rdd2022es`), turunan RDD2022, lisensi **CC BY-NC-SA 4.0** — boleh untuk tugas akademik, bukan komersial. Sertakan sitasi Arya dkk. (2022) di laporan.


## 1. Cek lingkungan

In [ ]:
#@title 1) Cek Python, TensorFlow, dan GPU
import platform, sys

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())

try:
    import tensorflow as tf
except ImportError as exc:
    raise SystemExit("TensorFlow tidak ditemukan. Di Colab: Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.") from exc

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU       :", gpus if gpus else "TIDAK ADA -> training akan lambat. Aktifkan GPU di menu Runtime.")

!nvidia-smi -L 2>/dev/null || true


## 2. Konfigurasi pipeline

Semua parameter yang mungkin perlu diubah ada di satu sel ini. Kalau ingin memakai dataset lain, cukup ganti `KAGGLE_DATASET` dan sesuaikan `SEVERITY_MAP`.


In [ ]:
#@title 2) Konfigurasi
import os

# --- Sumber data ---
KAGGLE_DATASET = "juusos/rdd2022es"   # RDD2022ES: 8 kelas dengan tingkat keparahan
RAW_DIR        = "/content/raw"

# --- Folder kerja ---
WORK_DIR  = "/content/snaproad"
CROPS_DIR = f"{WORK_DIR}/crops"       # hasil crop bbox -> 3 kelas
MODEL_DIR = f"{WORK_DIR}/model_out"   # .tflite + metrik + grafik
for d in (RAW_DIR, WORK_DIR, CROPS_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

# --- 3 kelas aplikasi (urutan ini WAJIB sama dengan labels.txt dan Flutter) ---
CLASS_ORDER = ["ringan", "sedang", "berat"]

# --- Pemetaan nama kelas dataset -> 3 tingkat kerusakan ---
SEVERITY_MAP = {
    "longitudinal crack":      "ringan",
    "transverse crack":        "ringan",
    "longitudinal crack wide": "sedang",
    "transverse crack wide":   "sedang",
    "alligator crack":         "sedang",
    "pothole":                 "sedang",
    "alligator crack sunken":  "berat",
    "pothole deep":            "berat",
}

# --- Parameter data & training ---
IMG_SIZE            = 224
BATCH_SIZE          = 32
SEED                = 42
CROP_MARGIN         = 0.20    # perluasan kotak bbox saat crop (20% sisi)
MIN_CROP_PX         = 48      # bbox lebih kecil dari ini dibuang
TRAIN_CAP_PER_CLASS = 3000    # batas crop per kelas (menjaga waktu training Colab)
VALID_CAP_PER_CLASS = 600
TEST_CAP_PER_CLASS  = 600

print("Kelas aplikasi :", CLASS_ORDER)
print("Folder kerja   :", WORK_DIR)


## 3. Unduh dataset dari Kaggle

Butuh file `kaggle.json` (token API). Notebook akan meminta unggah sekali, lalu menyimpannya di `~/.kaggle/`.
Kalau dataset sudah pernah diunduh, sel ini otomatis dilewati.


In [ ]:
#@title 3) Unduh RDD2022ES dari Kaggle
import os, shutil, subprocess, sys

DONE_FLAG = os.path.join(RAW_DIR, "_DOWNLOADED")

if not os.path.exists(DONE_FLAG):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

    # 1) Pastikan token tersedia
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    token_dst  = os.path.join(kaggle_dir, "kaggle.json")
    token_src  = "/content/kaggle.json"

    if not os.path.exists(token_dst) and not os.path.exists(token_src):
        try:
            from google.colab import files
            print("Unggah kaggle.json (Kaggle > Settings > API > Create New Token):")
            uploaded = files.upload()
            token_src = next(iter(uploaded))
        except Exception as exc:
            raise SystemExit(
                "kaggle.json belum ada. Letakkan di /content/kaggle.json "
                "(atau ~/.kaggle/kaggle.json) lalu jalankan sel ini lagi."
            ) from exc

    if not os.path.exists(token_dst):
        shutil.copy(token_src, token_dst)
    os.chmod(token_dst, 0o600)

    # 2) Unduh + ekstrak
    print(f"Mengunduh {KAGGLE_DATASET} (~2,6 GB), mohon tunggu ...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", RAW_DIR, "--unzip"],
        check=True,
    )
    open(DONE_FLAG, "w").write("ok")
    print("Selesai.")
else:
    print("Dataset sudah ada, lanjut.")

total = sum(f.stat().st_size for f in __import__("pathlib").Path(RAW_DIR).rglob("*") if f.is_file())
print(f"Total ukuran di {RAW_DIR}: {total/1e9:.2f} GB")


## 4. Periksa struktur dataset

Sel ini sengaja mencetak struktur folder dan contoh isi label. Tujuannya: memastikan asumsi parser cocok dengan dataset yang benar-benar terunduh. Kalau formatnya berbeda (mis. XML Pascal VOC), parser di langkah berikut sudah menangani dua-duanya.


In [ ]:
#@title 4) Periksa struktur dataset
from pathlib import Path
from collections import Counter

RAW = Path(RAW_DIR)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

images = [p for p in RAW.rglob("*") if p.suffix.lower() in IMG_EXTS]
txts   = [p for p in RAW.rglob("*.txt")]
xmls   = [p for p in RAW.rglob("*.xml")]
yamls  = list(RAW.rglob("*.yaml")) + list(RAW.rglob("*.yml"))

print(f"Gambar : {len(images):,}")
print(f"Label  : {len(txts):,} (.txt) + {len(xmls):,} (.xml)")
print(f"YAML   : {len(yamls)}")

print("\nFolder gambar terbanyak:")
for folder, n in Counter(p.parent.relative_to(RAW).as_posix() for p in images).most_common(25):
    print(f"  {n:>7,}  {folder}")

if yamls:
    print("\nIsi data.yaml:", yamls[0])
    print(yamls[0].read_text(errors="ignore")[:900])

if txts:
    sample = txts[0]
    print(f"\nContoh label: {sample.relative_to(RAW)}")
    print("\n".join(sample.read_text(errors="ignore").splitlines()[:5]))


## 5. Deteksi nama kelas & validasi pemetaan

Nama kelas dibaca dari `data.yaml`/`classes.txt` bila ada. Kalau tidak ada, dipakai urutan default RDD2022ES. Sel ini berhenti dengan pesan jelas kalau ada kelas yang belum punya pemetaan.


In [ ]:
#@title 5) Nama kelas + pemetaan ke 3 tingkat
import re
import yaml
from pathlib import Path

RAW = Path(RAW_DIR)

def _norm(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def find_class_names(root):
    for yml in list(root.rglob("data.yaml")) + list(root.rglob("*.yml")):
        try:
            data = yaml.safe_load(yml.read_text(encoding="utf-8", errors="ignore"))
        except Exception:
            continue
        names = data.get("names") if isinstance(data, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names, key=lambda k: int(k))]
        if names:
            return [_norm(n) for n in names], yml
    for txt in list(root.rglob("classes.txt")) + list(root.rglob("obj.names")):
        names = [_norm(line) for line in txt.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
        if names:
            return names, txt
    return None, None

CLASS_NAMES, names_file = find_class_names(RAW)
if CLASS_NAMES is None:
    CLASS_NAMES = [
        "longitudinal crack", "longitudinal crack wide",
        "transverse crack", "transverse crack wide",
        "alligator crack", "alligator crack sunken",
        "pothole", "pothole deep",
    ]
    print("File nama kelas tidak ditemukan -> memakai urutan default RDD2022ES.")
else:
    print("Nama kelas dibaca dari:", names_file)

def severity_for(name):
    n = _norm(name).replace("_", " ").replace("-", " ")
    if n in SEVERITY_MAP:
        return SEVERITY_MAP[n]
    # Cadangan kalau penulisan nama sedikit berbeda
    if "deep" in n or "sunken" in n:
        return "berat"
    if "wide" in n or "alligator" in n or "pothole" in n:
        return "sedang"
    if "crack" in n:
        return "ringan"
    return None

ID_TO_SEVERITY = {}
for idx, name in enumerate(CLASS_NAMES):
    sev = severity_for(name)
    if sev is None:
        raise SystemExit(
            f"Kelas '{name}' (id {idx}) belum punya pemetaan. "
            "Tambahkan dulu di SEVERITY_MAP pada sel Konfigurasi, lalu jalankan ulang."
        )
    ID_TO_SEVERITY[idx] = sev

CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
print()
for idx, name in enumerate(CLASS_NAMES):
    print(f"  id {idx}  {name:<26} -> {ID_TO_SEVERITY[idx]}")


## 6. Bangun dataset klasifikasi (crop per bbox)

RDD2022ES berisi foto jalan **utuh** dengan anotasi kotak (YOLO). Model klasifikasi butuh gambar per kerusakan, jadi setiap bbox dipotong (dengan margin 20%) lalu di-resize ke 224×224.

Tiga hal penting:

1. **Anti-kebocoran data.** Crop dari foto yang sama tidak boleh masuk train dan test sekaligus. Foto juga ada versi cermin (mirror), jadi pengelompokan memakai *perceptual hash* yang tahan pencerminan.
2. **Pembagian data per grup**, bukan per file: ±70% train, 15% validasi, 15% test.
3. **Batas jumlah crop per kelas** supaya training di Colab tetap wajar.


In [ ]:
#@title 6) Crop bbox -> folder train/val/test
import csv, hashlib, random, time
import numpy as np
from collections import defaultdict
from pathlib import Path
from PIL import Image, ImageOps

random.seed(SEED)
np.random.seed(SEED)

def mirror_hash(img):
    """Deskriptor 16x16 yang kebal pencerminan horizontal.

    Nilai tiap baris diurutkan (urutan kolom diabaikan) lalu dikuantisasi
    kasar, sehingga gambar dan versi cerminnya menghasilkan hash yang sama.
    """
    gray = np.asarray(img.convert("L").resize((16, 16), Image.Resampling.BILINEAR), dtype=np.float32)
    rows = np.sort(gray, axis=1)
    quantized = (rows // 64).astype(np.uint8)
    return hashlib.md5(quantized.tobytes()).hexdigest()

def split_for(key):
    bucket = int(key[:8], 16) % 100
    if bucket < 15:
        return "test"
    if bucket < 30:
        return "val"
    return "train"

def expand_box(x1, y1, x2, y2, W, H, margin):
    bw, bh = x2 - x1, y2 - y1
    if bw < 4 or bh < 4:
        return None
    mx, my = bw * margin, bh * margin
    box = (int(max(0, x1 - mx)), int(max(0, y1 - my)),
           int(min(W, x2 + mx)), int(min(H, y2 + my)))
    if box[2] - box[0] < MIN_CROP_PX or box[3] - box[1] < MIN_CROP_PX:
        return None
    return box

def parse_yolo(txt_path, W, H):
    boxes = []
    for line in txt_path.read_text(errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(float(parts[0]))
        vals = [float(v) for v in parts[1:]]
        if len(vals) == 4:                       # cx cy w h
            cx, cy, w, h = vals
            x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
        else:                                    # poligon segmentasi -> bbox
            xs, ys = vals[0::2], vals[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        if max(x1, y1, x2, y2) <= 1.5:           # koordinat ternormalisasi
            x1, x2 = x1 * W, x2 * W
            y1, y2 = y1 * H, y2 * H
        boxes.append((cid, x1, y1, x2, y2))
    return boxes

import xml.etree.ElementTree as ET

def parse_voc(xml_path):
    boxes = []
    try:
        root_xml = ET.parse(xml_path).getroot()
    except Exception:
        return boxes
    for obj in root_xml.findall("object"):
        name = (obj.findtext("name") or "").strip().lower()
        if name not in CLASS_INDEX:
            continue
        bb = obj.find("bndbox")
        if bb is None:
            continue
        try:
            boxes.append((CLASS_INDEX[name],
                          float(bb.findtext("xmin")), float(bb.findtext("ymin")),
                          float(bb.findtext("xmax")), float(bb.findtext("ymax"))))
        except (TypeError, ValueError):
            continue
    return boxes

# --- Indeks label ---
images = [p for p in RAW.rglob("*") if p.suffix.lower() in IMG_EXTS]
txt_by_stem = defaultdict(list)
for t in RAW.rglob("*.txt"):
    txt_by_stem[t.stem].append(t)
xml_by_stem = defaultdict(list)
for x in RAW.rglob("*.xml"):
    xml_by_stem[x.stem].append(x)

def find_label(img):
    for by_stem in (txt_by_stem, xml_by_stem):
        cands = by_stem.get(img.stem, [])
        if not cands:
            continue
        for c in cands:                       # sefolder
            if c.parent == img.parent:
                return c
        for c in cands:                       # images/ vs labels/
            if c.parent.parent == img.parent.parent:
                return c
        return cands[0]
    return None

pairs = [(img, find_label(img)) for img in images]
pairs = [(img, lbl) for img, lbl in pairs if lbl is not None]
print(f"Gambar total        : {len(images):,}")
print(f"Punya anotasi       : {len(pairs):,}")
if not pairs:
    raise SystemExit("Tidak ada pasangan gambar+label. Cek hasil sel 'Periksa struktur dataset'.")

# --- Tulis crop ---
CAPS = {"train": TRAIN_CAP_PER_CLASS, "val": VALID_CAP_PER_CLASS, "test": TEST_CAP_PER_CLASS}
counts = {split: {cls: 0 for cls in CLASS_ORDER} for split in CAPS}
errors = []
manifest_rows = []
random.shuffle(pairs)

t0 = time.time()
for i, (img_path, lbl_path) in enumerate(pairs, 1):
    try:
        with Image.open(img_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            W, H = im.size
            split = split_for(mirror_hash(im))
            if all(counts[split][c] >= CAPS[split] for c in CLASS_ORDER):
                continue

            boxes = parse_yolo(lbl_path, W, H) if lbl_path.suffix.lower() == ".txt" else parse_voc(lbl_path)
            for b_idx, (cid, x1, y1, x2, y2) in enumerate(boxes):
                sev = ID_TO_SEVERITY.get(cid)
                if sev is None or counts[split][sev] >= CAPS[split]:
                    continue
                box = expand_box(x1, y1, x2, y2, W, H, CROP_MARGIN)
                if box is None:
                    continue

                crop = im.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
                out_dir = Path(CROPS_DIR) / split / sev
                out_dir.mkdir(parents=True, exist_ok=True)
                tag = hashlib.md5(str(img_path).encode()).hexdigest()[:8]
                fname = f"{img_path.stem[:40]}_{tag}_{b_idx:02d}.jpg"
                crop.save(out_dir / fname, quality=92)

                counts[split][sev] += 1
                manifest_rows.append([img_path.relative_to(RAW).as_posix(), split, sev,
                                      *[round(v, 1) for v in box]])
    except Exception as exc:
        errors.append(f"{img_path}: {exc}")

    if i % 500 == 0:
        print(f"  {i:,}/{len(pairs):,} gambar | "
              f"train {sum(counts['train'].values()):,} | "
              f"val {sum(counts['val'].values()):,} | "
              f"test {sum(counts['test'].values()):,} | {time.time()-t0:.0f}s")

manifest_path = Path(WORK_DIR) / "manifest_crops.csv"
with open(manifest_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source", "split", "kelas", "x1", "y1", "x2", "y2"])
    writer.writerows(manifest_rows)

print(f"\nSelesai dalam {time.time()-t0:.0f} detik. Gagal dibaca: {len(errors)}")
print("Manifest:", manifest_path)
for split in ("train", "val", "test"):
    print(f"{split:>5}: " + " | ".join(f"{c}={counts[split][c]:,}" for c in CLASS_ORDER))


## 7. Distribusi kelas & contoh crop

Cek apakah kelas cukup seimbang. Kalau salah satu kelas jauh lebih sedikit, laporkan sebagai batasan dan andalkan `class_weight` saat training.


In [ ]:
#@title 7) EDA: distribusi kelas + contoh gambar
import random
from pathlib import Path
import matplotlib.pyplot as plt

counts = {}
for split in ("train", "val", "test"):
    counts[split] = {cls: len(list((Path(CROPS_DIR) / split / cls).glob("*.jpg"))) for cls in CLASS_ORDER}

fig, ax = plt.subplots(figsize=(8, 3.5))
width = 0.25
x = range(len(CLASS_ORDER))
for offset, split in zip((-width, 0, width), ("train", "val", "test")):
    ax.bar([v + offset for v in x], [counts[split][c] for c in CLASS_ORDER], width, label=split)
ax.set_xticks(list(x))
ax.set_xticklabels(CLASS_ORDER)
ax.set_ylabel("jumlah crop")
ax.set_title("Distribusi kelas per split")
ax.legend()
plt.tight_layout()
plt.savefig(Path(MODEL_DIR) / "class_distribution.png", dpi=150)
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for row, cls in enumerate(CLASS_ORDER):
    files = list((Path(CROPS_DIR) / "train" / cls).glob("*.jpg"))
    for col, ax in enumerate(axes[row]):
        ax.axis("off")
        if col < len(files):
            ax.imshow(plt.imread(random.choice(files)))
        if col == 0:
            ax.set_title(cls, loc="left")
plt.suptitle("Contoh crop tiap kelas (train)")
plt.tight_layout()
plt.show()


## 8. Pipeline `tf.data`

Model menerima **float32 0–255**, lalu menormalisasi sendiri di dalam grafik (`Rescaling → [-1, 1]`). Jadi di Flutter nanti cukup kirim nilai piksel mentah tanpa membagi 255 — ini mencegah bug preprocessing yang paling sering terjadi.


In [ ]:
#@title 8) Dataset TensorFlow
import numpy as np
import tensorflow as tf
from pathlib import Path

AUTOTUNE = tf.data.AUTOTUNE

def list_split(split):
    files, labels = [], []
    for idx, cls in enumerate(CLASS_ORDER):
        for p in sorted((Path(CROPS_DIR) / split / cls).glob("*.jpg")):
            files.append(str(p))
            labels.append(idx)
    return files, labels

train_files, train_labels = list_split("train")
val_files,   val_labels   = list_split("val")
test_files,  test_labels  = list_split("test")
print(f"train={len(train_files):,}  val={len(val_files):,}  test={len(test_files):,}")
assert train_files and val_files and test_files, "Ada split yang kosong. Jalankan ulang sel crop."

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)            # 0..255, TIDAK dibagi 255
    return img, tf.one_hot(label, len(CLASS_ORDER))

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.12 * 255)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 255.0), label

def make_ds(files, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if training:
        ds = ds.shuffle(min(len(files), 10000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_files, train_labels, training=True)
val_ds   = make_ds(val_files, val_labels)
test_ds  = make_ds(test_files, test_labels)

# class_weight untuk data tidak seimbang
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_ORDER)), y=np.array(train_labels))
class_weight = {i: float(w) for i, w in enumerate(weights)}
print("class_weight:", {CLASS_ORDER[k]: round(v, 3) for k, v in class_weight.items()})


## 9. Model: MobileNetV2 + transfer learning

Dua tahap:

1. **Frozen base** — kepala klasifikasi dilatih dulu (lr 1e-3).
2. **Fine-tuning** — 40 layer terakhir MobileNetV2 dibuka dengan lr kecil (1e-5). BatchNormalization tetap dibekukan agar statistiknya stabil.


In [ ]:
#@title 9) Bangun model
from tensorflow import keras

def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype="float32", name="image")
    x = keras.layers.Rescaling(1 / 127.5, offset=-1.0, name="preprocess")(inputs)

    base = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False

    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(128, activation="relu", name="fc")(x)
    x = keras.layers.Dropout(0.2)(x)
    outputs = keras.layers.Dense(len(CLASS_ORDER), activation="softmax", name="severity")(x)

    return keras.Model(inputs, outputs, name="snaproad_mobilenetv2"), base

model, base_model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:
#@title 10) Tahap 1 — latih kepala klasifikasi (base dibekukan)
from pathlib import Path

callbacks_1 = [
    keras.callbacks.ModelCheckpoint(str(Path(MODEL_DIR) / "stage1_best.keras"),
                                    monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2,
                                      min_lr=1e-6, verbose=1),
]

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weight,
    callbacks=callbacks_1,
)


In [ ]:
#@title 11) Tahap 2 — fine-tuning 40 layer terakhir
for layer in base_model.layers[:-40]:
    layer.trainable = False
for layer in base_model.layers:
    layer.trainable = True
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_2 = [
    keras.callbacks.ModelCheckpoint(str(Path(MODEL_DIR) / "best_model.keras"),
                                    monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2,
                                      min_lr=1e-7, verbose=1),
]

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks_2,
)


## 10. Evaluasi pada data uji

Target di dokumen: **akurasi ≥ 80%** dan **macro F1 ≥ 0,75**. Kalau belum tercapai, lihat confusion matrix: biasanya kelas `ringan` dan `sedang` paling sering tertukar. Perbaikan yang bisa dicoba: tambah epoch fine-tuning, naikkan batas crop per kelas, atau perbaiki pemetaan kelas.


In [ ]:
#@title 12) Evaluasi: akurasi, F1 makro, confusion matrix
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)

y_true = np.concatenate([y.numpy() for _, y in test_ds]).argmax(axis=1)
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")
report = classification_report(y_true, y_pred, target_names=CLASS_ORDER, digits=4)
print(report)
print(f"Akurasi     : {acc:.4f}")
print(f"F1 makro    : {f1_macro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_ORDER)), CLASS_ORDER)
ax.set_yticks(range(len(CLASS_ORDER)), CLASS_ORDER)
ax.set_xlabel("prediksi")
ax.set_ylabel("aktual")
for i in range(len(CLASS_ORDER)):
    for j in range(len(CLASS_ORDER)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title("Confusion matrix (test)")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(Path(MODEL_DIR) / "confusion_matrix.png", dpi=150)
plt.show()

metrics = {
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "class_order": CLASS_ORDER,
    "per_class": classification_report(y_true, y_pred, target_names=CLASS_ORDER,
                                       output_dict=True),
    "n_test": int(len(y_true)),
}
(Path(MODEL_DIR) / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("Metrik disimpan:", Path(MODEL_DIR) / "metrics.json")


In [ ]:
#@title 13) Kurva training
import matplotlib.pyplot as plt
from pathlib import Path

def concat(hist, key):
    return hist.history.get(key, [])

acc_1, acc_2 = concat(history_1, "accuracy"), concat(history_2, "accuracy")
val_1, val_2 = concat(history_1, "val_accuracy"), concat(history_2, "val_accuracy")
loss_1, loss_2 = concat(history_1, "loss"), concat(history_2, "loss")
vloss_1, vloss_2 = concat(history_1, "val_loss"), concat(history_2, "val_loss")

epochs = range(1, len(acc_1) + len(acc_2) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, acc_1 + acc_2, marker="o", label="train")
axes[0].plot(epochs, val_1 + val_2, marker="s", label="validasi")
axes[0].axvline(len(acc_1) + 0.5, color="gray", ls="--")
axes[0].set_title("Akurasi per epoch")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, loss_1 + loss_2, marker="o", label="train")
axes[1].plot(epochs, vloss_1 + vloss_2, marker="s", label="validasi")
axes[1].axvline(len(loss_1) + 0.5, color="gray", ls="--")
axes[1].set_title("Loss per epoch")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.suptitle("Garis putus-putus = mulai fine-tuning")
plt.tight_layout()
plt.savefig(Path(MODEL_DIR) / "training_curves.png", dpi=150)
plt.show()


## 11. Ekspor ke TensorFlow Lite

Tiga varian:

| Varian | Perkiraan ukuran | Kegunaan |
|---|---|---|
| `fp32` | ±9 MB | pembanding (bukan untuk aplikasi) |
| `fp16` | ±2,5 MB | **dipakai di aplikasi** (akurasi hampir sama dengan fp32) |
| `int8` | ±0,8 MB | eksperimen kuantisasi untuk laporan |

Ekspor lewat `SavedModel` (`model.export`) karena paling stabil di TensorFlow 2.16+ dengan Keras 3.


In [ ]:
#@title 14) Konversi ke .tflite
import shutil
from pathlib import Path

def convert_to_tflite(quant):
    """quant: 'fp32' | 'fp16' | 'int8'"""
    saved_dir = Path(MODEL_DIR) / f"saved_{quant}"
    if saved_dir.exists():
        shutil.rmtree(saved_dir)
    model.export(str(saved_dir))

    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_dir))
    if quant == "fp16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif quant == "int8":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = rep_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.float32

    blob = converter.convert()
    out_path = Path(MODEL_DIR) / f"snaproad_mobilenetv2_{quant}.tflite"
    out_path.write_bytes(blob)
    print(f"{quant:<4} -> {out_path.name:<40} {out_path.stat().st_size/1e6:6.2f} MB")
    return out_path

# Sampel untuk kuantisasi INT8 (100 gambar train)
rep_imgs = np.concatenate([x.numpy() for x, _ in train_ds.take(4)], axis=0)[:100]
def rep_gen():
    for i in range(len(rep_imgs)):
        yield [rep_imgs[i:i + 1].astype(np.float32)]

tflite_fp32 = convert_to_tflite("fp32")
tflite_fp16 = convert_to_tflite("fp16")
try:
    tflite_int8 = convert_to_tflite("int8")
except Exception as exc:
    tflite_int8 = None
    print("Konversi INT8 gagal (tidak masalah, pakai fp16):", exc)


## 12. Verifikasi model TFLite

Yang dicek: (1) bentuk & tipe input/output, (2) kesesuaian prediksi TFLite vs Keras, (3) waktu inferensi. Ini juga menjadi bukti di laporan bahwa model yang diekspor benar-benar mereproduksi hasil training.


In [ ]:
#@title 15) Uji TFLite: detail tensor, kesesuaian, latensi
import time
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

def tflite_interpreter(path):
    interp = tf.lite.Interpreter(model_path=str(path))
    interp.allocate_tensors()
    return interp

def tflite_predict(interp, img_0_255):
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    x = img_0_255
    if inp["dtype"] == np.uint8:
        x = np.clip(x, 0, 255).astype(np.uint8)
    else:
        x = x.astype(np.float32)
    interp.set_tensor(inp["index"], np.expand_dims(x, axis=0))
    interp.invoke()
    return interp.get_tensor(out["index"])[0]

interp = tflite_interpreter(tflite_fp16)
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print("Input :", inp["name"], inp["shape"], inp["dtype"])
print("Output:", out["name"], out["shape"], out["dtype"])

sample = test_files[:60]
imgs = np.stack([img_to_array(load_img(f, target_size=(IMG_SIZE, IMG_SIZE))) for f in sample])
keras_prob = model.predict(imgs, verbose=0)
tflite_prob = np.stack([tflite_predict(interp, im) for im in imgs])

agreement = (keras_prob.argmax(1) == tflite_prob.argmax(1)).mean()
max_delta = float(np.abs(keras_prob - tflite_prob).max())
print(f"Kesesuaian prediksi Keras vs TFLite : {agreement*100:.2f}%")
print(f"Selisih probabilitas maksimum       : {max_delta:.5f}")

latencies = []
for im in imgs[:30]:
    start = time.perf_counter()
    tflite_predict(interp, im)
    latencies.append((time.perf_counter() - start) * 1000)
print(f"Latensi (CPU Colab, 30 run)         : {np.mean(latencies):.1f} ms "
      f"(median {np.median(latencies):.1f} ms)")
print("Catatan: latensi di HP bisa berbeda; ukur juga di perangkat saat pengujian aplikasi.")


## 13. Simpan artefak untuk repo & Flutter

Isi `snaproad_model.zip`: model fp16 (utama), int8, labels, metadata, metrik, dan grafik. Simpan hasil unduhan ke `ml/artifacts/` lalu commit **tanpa** dataset mentah.


In [ ]:
#@title 16) labels.txt, model_meta.json, dan unduh hasil
import json, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path

OUT = Path(MODEL_DIR)

# labels.txt — urutan HARUS sama dengan CLASS_ORDER
(OUT / "labels.txt").write_text("\n".join(CLASS_ORDER) + "\n", encoding="utf-8")

meta = {
    "model": "MobileNetV2 (ImageNet) + transfer learning",
    "input": {"size": IMG_SIZE, "dtype": "float32", "range": "0-255 (model menormalisasi internal ke [-1,1])"},
    "class_order": CLASS_ORDER,
    "dataset": {
        "kaggle": KAGGLE_DATASET,
        "license": "CC BY-NC-SA 4.0",
        "mapping": SEVERITY_MAP,
        "train": counts["train"],
        "val": counts["val"],
        "test": counts["test"],
    },
    "metrics": {
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
    },
    "files": {
        "main": "snaproad_mobilenetv2_fp16.tflite",
        "int8": "snaproad_mobilenetv2_int8.tflite" if tflite_int8 else None,
    },
    "tflite_inference": {
        "input_dtype": str(inp["dtype"]),
        "keras_tflite_agreement": float(agreement),
        "cpu_latency_ms": float(np.mean(latencies)),
    },
    "versions": {
        "tensorflow": tf.__version__,
        "python": ".".join(map(str, __import__("sys").version_info[:3])),
    },
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(OUT / "model_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

# Ringkasan ukuran
print("Artefak:")
for p in sorted(OUT.glob("*.tflite")) + [OUT / "labels.txt", OUT / "model_meta.json",
                                         OUT / "metrics.json", OUT / "confusion_matrix.png",
                                         OUT / "training_curves.png", OUT / "class_distribution.png"]:
    if p.exists():
        print(f"  {p.name:<44} {p.stat().st_size/1024:8.1f} KB")

# Arsip untuk diunduh
zip_path = Path("/content/snaproad_model.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.glob("*"):
        if p.is_file() and not p.name.endswith(".keras"):
            zf.write(p, arcname=p.name)
print(f"\nArsip: {zip_path} ({zip_path.stat().st_size/1e6:.2f} MB)")

# Opsional: simpan ke Google Drive agar tidak hilang saat runtime berakhir
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    drive_dir = Path("/content/drive/MyDrive/SNAPROAD")
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, drive_dir / "snaproad_model.zip")
    print("Salinan di Drive:", drive_dir / "snaproad_model.zip")
    files.download(str(zip_path))
except Exception as exc:
    print("Lewati unduh otomatis / Drive:", exc)


## 14. Selesai — langkah berikutnya

**Yang harus dilakukan setelah notebook ini jalan:**

1. Unduh `snaproad_model.zip` → ekstrak ke `ml/artifacts/`.
2. Commit bertahap (contoh pesan):
   - `feat(ml): tambah notebook training MobileNetV2`
   - `feat(ml): ekspor model TFLite fp16 dan int8`
   - `docs(ml): catat hasil evaluasi dan confusion matrix`
   - `chore: tambahkan dataset dan artefacts ke .gitignore`
3. Tempel angka `accuracy` dan `f1_macro` dari `metrics.json` ke laporan.
4. Lanjut ke tahap Flutter: `assets/models/snaproad_mobilenetv2_fp16.tflite` + `assets/models/labels.txt`.

**Kalau hasil evaluasi masih di bawah target**, urutan perbaikan yang paling efektif:

1. Naikkan `TRAIN_CAP_PER_CLASS` ke 5000–6000 (butuh waktu training lebih lama).
2. Tambah epoch fine-tuning dan turunkan `patience` EarlyStopping.
3. Periksa confusion matrix: jika `ringan`↔`sedang` tertukar, pertimbangkan memindahkan `alligator crack` ke `ringan` atau `berat` dan jelaskan di laporan.
4. Pastikan tidak ada kebocoran data (mirror sudah ditangani lewat hash).
